In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Read the CSV file
df = pd.read_csv('summary_last2.csv')

# Separate main experiments and external validation
df_main = df[~df['experiment'].str.contains('external', na=False)]
df_external = df[df['experiment'].str.contains('external', na=False)]

# Create pivot tables for balanced accuracy and standard deviation (main experiments)
plot_data = df_main.pivot_table(
    index='dataset', 
    columns='aug_label', 
    values='ensemble_bal_acc'
)

plot_data_std = df_main.pivot_table(
    index='dataset', 
    columns='aug_label', 
    values='fold_bal_acc_std'
)

# Get external validation data for dermamnist-e
external_data = df_external[df_external['dataset'] == 'dermamnist-e external'].set_index('aug_label')['ensemble_bal_acc']
external_std = df_external[df_external['dataset'] == 'dermamnist-e external'].set_index('aug_label')['fold_bal_acc_std']

# Create the barplot
fig, ax = plt.subplots(figsize=(14, 6))

# Plot main data
x = np.arange(len(plot_data.index))
width = 0.35
offset = 0.1  # Offset for external validation bars

bars1 = ax.bar(x - width/2, plot_data['no aug'], width, label='no aug', alpha=0.8,
               yerr=plot_data_std['no aug'], capsize=5, error_kw={'linewidth': 1.5})
bars2 = ax.bar(x + width/2, plot_data['aug'], width, label='aug', alpha=0.8,
               yerr=plot_data_std['aug'], capsize=5, error_kw={'linewidth': 1.5})

# Add external validation bars for dermamnist-e (slightly offset)
dermamnist_e_idx = list(plot_data.index).index('dermamnist-e')
if 'no aug' in external_data.index:
    ax.bar(dermamnist_e_idx - width/2 - offset, external_data['no aug'], width*0.7, 
           color='lightblue', alpha=0.5, hatch='//', label='external no aug',
           yerr=external_std['no aug'], capsize=5, error_kw={'linewidth': 1.5})
if 'aug' in external_data.index:
    ax.bar(dermamnist_e_idx + width/2 + offset, external_data['aug'], width*0.7, 
           color='lightcoral', alpha=0.5, hatch='//', label='external aug',
           yerr=external_std['aug'], capsize=5, error_kw={'linewidth': 1.5})

ax.set_xlabel('Dataset', fontsize=12)
ax.set_ylabel('Balanced Accuracy', fontsize=12)
ax.set_title('Ensembling Balanced Accuracy Results Across Datasets (SD from 5CV)', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(plot_data.index, rotation=45, ha='right')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
ax.set_ylim([0.5, 1.0])

plt.tight_layout()
plt.show()

# Display the exact values
print("\nBalanced Accuracy Values (Main Experiments):")
print(plot_data.round(4))
print("\nStandard Deviation Values (Main Experiments):")
print(plot_data_std.round(4))
print("\nBalanced Accuracy Values (External Validation - dermamnist-e):")
print(external_data.round(4))
print("\nStandard Deviation Values (External Validation - dermamnist-e):")
print(external_std.round(4))